# 03 - Price Impact Analysis

**The core research question: does OFI predict future price changes?**

1. OFI → forward return regression with Newey-West standard errors
2. Multi-horizon analysis - how long does OFI signal persist?
3. R² decomposition by LOB level
4. Rolling regression - time-varying impact
5. Permanent vs temporary price impact
6. Best-level vs integrated OFI comparison

In [1]:
import importlib.util, os, pickle
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, os.path.abspath(path))
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

pi_mod  = load_module('price_impact', '../src/price_impact.py')
viz_mod = load_module('viz',          '../src/visualization.py')

with open('../data/ofi_data.pkl', 'rb') as f:
    data = pickle.load(f)

df         = data['df']
level_cols = data['level_cols']
agg_5s     = data['agg_results']['5s']   # use 5s for cleaner signal

print(f'Loaded {len(df)} tick events')
print(f'5s aggregated: {len(agg_5s)} intervals')

Loaded 5000 tick events
5s aggregated: 643 intervals


## 1. OFI Price Impact - Main Regression

In [2]:
print('=== OFI PRICE IMPACT REGRESSION ===')
print('Model: Δp_{t+1} = α + β × OFI_t + ε_t')
print('Standard errors: Newey-West HAC (corrects for autocorrelation)')
print()

for ofi_col in ['ofi_best', 'ofi_integrated']:
    if ofi_col not in agg_5s.columns:
        continue
    result = pi_mod.ofi_price_impact(agg_5s, ofi_col=ofi_col, return_col='fwd_return')
    if 'error' not in result:
        print(f'{ofi_col.upper()}:')
        print(f'  β (impact):        {result["beta"]:.8f}')
        print(f'  t-stat (NW):       {result["t_stat_nw"]:.3f}')
        print(f'  p-value (NW):      {result["p_value_nw"]:.4f}')
        print(f'  R²:                {result["r_squared"]:.4f}')
        print(f'  Significant (5%):  {result["significant"]}')
        print(f'  N observations:    {result["n_obs"]}')
        print()

=== OFI PRICE IMPACT REGRESSION ===
Model: Δp_{t+1} = α + β × OFI_t + ε_t
Standard errors: Newey-West HAC (corrects for autocorrelation)

OFI_BEST:
  β (impact):        -0.00000000
  t-stat (NW):       -0.149
  p-value (NW):      0.8813
  R²:                0.0001
  Significant (5%):  False
  N observations:    642

OFI_INTEGRATED:
  β (impact):        -0.00000143
  t-stat (NW):       -0.409
  p-value (NW):      0.6823
  R²:                0.0003
  Significant (5%):  False
  N observations:    642



## 2. Multi-Horizon Analysis

In [3]:
print('Running multi-horizon regression...')
horizons = [1, 2, 3, 5, 10, 15, 20, 30]

horizon_results = {}
for ofi_col in ['ofi_best', 'ofi_integrated']:
    if ofi_col not in agg_5s.columns:
        continue
    h_df = pi_mod.multi_horizon_impact(agg_5s, ofi_col=ofi_col, horizons=horizons)
    horizon_results[ofi_col] = h_df
    print(f'\n{ofi_col}:')
    display(h_df)

Running multi-horizon regression...

ofi_best:


,horizon,beta,t_stat,p_value,r_squared,significant
0,1,-0.000000e+00,-0.152,0.8796,0.0001,False
1,2,-1.000000e-08,-0.513,0.6085,0.0009,False
2,3,-1.000000e-08,-0.402,0.6876,0.0005,False
3,5,-1.000000e-08,-0.375,0.7081,0.0005,False
4,10,-3.000000e-08,-0.925,0.3553,0.0023,False
5,15,-5.000000e-08,-1.246,0.2134,0.0041,False
6,20,-6.000000e-08,-2.022,0.0436,0.0049,True
7,30,-1.100000e-07,-3.276,0.0011,0.0120,True



ofi_integrated:


,horizon,beta,t_stat,p_value,r_squared,significant
0,1,-1.430000e-06,-0.401,0.6886,0.0003,False
1,2,-5.910000e-06,-1.446,0.1488,0.0024,False
2,3,-9.440000e-06,-1.720,0.0860,0.0044,False
3,5,-2.700000e-06,-0.390,0.6964,0.0002,False
4,10,1.000000e-07,0.009,0.9926,0.0000,False
5,15,1.026000e-05,0.685,0.4933,0.0011,False
6,20,1.178000e-05,0.710,0.4781,0.0011,False
7,30,-1.360000e-06,-0.074,0.9409,0.0000,False


In [4]:
if 'ofi_best' in horizon_results:
    fig = viz_mod.price_impact_horizon_chart(horizon_results['ofi_best'])
    fig.show()
    print('Interpretation:')
    print('  - β at h=1: immediate price impact per unit OFI')
    print('  - β decay: how quickly the signal loses predictive power')
    print('  - If β becomes 0 at large h: impact is temporary (noise)')
    print('  - If β stays positive: impact is permanent (information-driven)')

Interpretation:
  - β at h=1: immediate price impact per unit OFI
  - β decay: how quickly the signal loses predictive power
  - If β becomes 0 at large h: impact is temporary (noise)
  - If β stays positive: impact is permanent (information-driven)


## 3. R² Decomposition by LOB Level

In [5]:
# Which LOB levels predict price?
agg_levels = data['agg_results']['5s']
level_cols_in_agg = [c for c in level_cols if c in agg_levels.columns]

if level_cols_in_agg:
    level_r2 = pi_mod.level_r2_decomposition(
        agg_levels, level_cols_in_agg, return_col='fwd_return'
    )
    print('R² by LOB Level (predicting 1-period forward return):')
    display(level_r2)

    fig2 = viz_mod.level_r2_chart(level_r2)
    fig2.show()
    print('\nKey finding: If deeper levels add R², OFI carries information beyond the spread.')
    print('If only level 1 matters: shallow traders dominate price discovery.')

R² by LOB Level (predicting 1-period forward return):


,level,beta,r_squared,t_stat,p_value
6,ofi_level_06,-1.640000e-06,0.0051,-1.819,0.0693
2,ofi_level_02,1.350000e-06,0.0047,1.733,0.0836
3,ofi_level_03,-1.590000e-06,0.0044,-1.690,0.0915
5,ofi_level_05,1.180000e-06,0.0023,1.202,0.2296
8,ofi_level_08,-4.500000e-07,0.0004,-0.500,0.6172
0,ofi_level_00,-3.500000e-07,0.0001,-0.218,0.8275
1,ofi_level_01,-2.400000e-07,0.0001,-0.264,0.7921
4,ofi_level_04,-5.000000e-08,0.0000,-0.051,0.9594
7,ofi_level_07,-1.000000e-08,0.0000,-0.006,0.9950



Key finding: If deeper levels add R², OFI carries information beyond the spread.
If only level 1 matters: shallow traders dominate price discovery.


## 4. Rolling Regression - Time-Varying Impact

In [6]:
if 'ofi_best' in agg_5s.columns and 'fwd_return' in agg_5s.columns:
    rolling_df = pi_mod.rolling_price_impact(
        agg_5s, ofi_col='ofi_best', return_col='fwd_return', window=50
    )
    fig3 = viz_mod.rolling_impact_chart(rolling_df)
    fig3.show()
    print('Interpretation: Changing β over time indicates the market microstructure')
    print('is non-stationary. Impact may be higher near market open/close.')

Interpretation: Changing β over time indicates the market microstructure
is non-stationary. Impact may be higher near market open/close.


## 5. Permanent vs Temporary Impact

In [7]:
pt = pi_mod.permanent_temporary_impact(
    agg_5s, ofi_col='ofi_best',
    horizons=[1, 5, 10, 20, 30]
)

if pt:
    print('=== PERMANENT vs TEMPORARY PRICE IMPACT ===')
    print(f'Beta by horizon: {pt["betas_by_horizon"]}')
    print(f'Peak impact:     {pt["peak_impact"]:.8f}')
    print(f'Permanent:       {pt["permanent_impact"]:.8f}')
    print(f'Temporary:       {pt["temporary_impact"]:.8f}')
    print(f'Reversion ratio: {pt["reversion_ratio"]:.4f}')
    print()
    if pt['reversion_ratio'] > 0.5:
        print('> 50% of impact reverses → high noise/temporary component')
    else:
        print('< 50% of impact reverses → mostly permanent/information-driven')

    # Plot beta decay
    horizons = list(pt['betas_by_horizon'].keys())
    betas    = list(pt['betas_by_horizon'].values())
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(
        x=horizons, y=betas, mode='lines+markers',
        line=dict(color='#4C6EF5', width=2.5),
        marker=dict(size=8)
    ))
    fig4.add_hline(y=0, line_dash='dot', line_color='gray')
    fig4.update_layout(
        title='OFI Impact Decay — Permanent vs Temporary',
        xaxis_title='Horizon (5s intervals)', yaxis_title='β coefficient',
        height=400, template='plotly_white'
    )
    fig4.show()

=== PERMANENT vs TEMPORARY PRICE IMPACT ===
Beta by horizon: {1: np.float64(-0.0), 5: np.float64(-1e-08), 10: np.float64(-3e-08), 20: np.float64(-6e-08), 30: np.float64(-1.1e-07)}
Peak impact:     -0.00000000
Permanent:       -0.00000011
Temporary:       0.00000011
Reversion ratio: 1100.0000

> 50% of impact reverses → high noise/temporary component
